# 02 — Agent Definition (AI Engineer owns)

Assembles the single LLM + four tools into a LangGraph ReAct agent. The LLM reasons between every tool call; nothing is hard coded into a fixed sequence.

**Tools:** `eligibility_screener`, `practice_matcher`, `payment_estimator`, `deadline_lookup`. _Only `eligibility_screener` is bound so far (see `agent/graph.py` `TOOLS`); the other three are stubs being built separately._

Out of scope handling is not a tool. The system prompt (PTCF form, written for internal advisors) makes the agent decline irrelevant input and redirect, and drives a short elicitation flow that gathers the client's profile (state and county, acreage, current practices, primary resource concern) across turns.

## Setup

In [1]:
import asyncio
import sys

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [2]:
%load_ext autoreload
%autoreload 2

from nrcs_navigator import config
from nrcs_navigator.agent.graph import build_agent

# Prereqs: run notebook 01 first (payment_rates + eCFR vector store populated),
# the Postgres container is up, and OPENAI_API_KEY is set in .env -- both the
# premier model and the eligibility_screener query embedding use it.
print("imports ready")

imports ready


## Build the agent

In [3]:
# Build the ReAct agent on the premier model. model_name comes from config/.env;
# build_agent(config.CHEAP_MODEL) would build the same agent on the cheaper leg.
# See agent/graph.py for the list of TOOLS
agent = build_agent(config.PREMIER_MODEL)
print(f"agent built on {config.PREMIER_MODEL}")

agent built on gpt-4o


## Run an example query

In [9]:
# A realistic in-scope question from an advisor about a client. The agent reasons
# (ReAct), should call eligibility_screener, and answer with citations. thread_id
# keys this conversation in the Postgres checkpointer.
result = agent.invoke(
    {"messages": [("user",
        "Why is there no Estimated Payments Section for ACEP?")]},
    config={"configurable": {"thread_id": "jacob-test-1"}},
)

# Full ReAct trace: the question, any tool calls + tool output, then the answer.
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hi
================================== Ai Message ==================================

Hello! How can I assist you with NRCS conservation programs today?
================================ Human Message =================================

Soil erosion
================================== Ai Message ==================================

To help you identify suitable NRCS conservation programs for addressing soil erosion, I'll need a bit more information about your client's operation. Could you please provide the state where the operation is located?
================================ Human Message =================================

California
================================== Ai Message ==================================

Thank you. Could you also provide the approximate acreage of the operation?
================================ Human Message =================================

600
================================== 

## Demonstrate graceful rejection

In [10]:
# An out-of-scope request: CRP is administered by FSA, not NRCS. The scope guard
# in the system prompt should make the agent decline and redirect to the local
# FSA office WITHOUT calling any tool.
result = agent.invoke(
    {"messages": [("user",
        "Can you help my client enroll in the Conservation Reserve Program (CRP)?")]},
    config={"configurable": {"thread_id": "demo-crp"}},
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The Conservation Reserve Program (CRP) is administered by the Farm Service Agency (FSA), not the NRCS. Your client should contact their local FSA office for assistance with CRP enrollment.
